In [1]:
!pip install diffusers transformers accelerate datasets tqdm lpips

In [ ]:
## Training ##

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.utils import save_image, make_grid
from tqdm import tqdm
import os
from spnn_model import SPNNAutoencoder256

from spnn_model import (
    PixelUnshuffleBlock,
    ConvPINNBlock,
)

DEVICE = "cuda"
DATASET = "korexyz/celeba-hq-256x256"

BATCH_SIZE = 32
LR = 1e-4
NUM_EPOCHS = 150
SAVE_DIR = "checkpoints_spnn"
LOG_EVERY = 1000
SAVE_EVERY = 10


MIX_TYPE = "cayley"
SCALE_BOUND = 2.0
HIDDEN = 192
R_HIDDEN = 384

W_LATENT_L1 = 0.5
W_LATENT_MSE = 1.0
W_COSINE = 1.0
W_RECON = 0.5        
W_CYCLE = 0.1        
W_DECORR = 0.05        

PHASE1_EPOCHS = 80     

os.makedirs(SAVE_DIR, exist_ok=True)
torch.manual_seed(42)

transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(256),
    transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3),
])


class HFWrap(Dataset):
    def __init__(self, ds, n=None):
        self.ds = ds
        self.n = n if n else len(ds)
    def __len__(self):
        return min(self.n, len(self.ds))
    def __getitem__(self, idx):
        img = self.ds[idx]["image"]
        if img.mode != "RGB": img = img.convert("RGB")
        return transform(img)

def get_loaders():
    from datasets import load_dataset
    ds = load_dataset(DATASET, split="train")
    n = len(ds)
    n_train = int(0.95 * n)
    train_loader = DataLoader(HFWrap(ds.select(range(n_train))),
                              batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=True)
    val_loader = DataLoader(HFWrap(ds.select(range(n_train, n))),
                            batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)
    return train_loader, val_loader


def load_vqvae():
    from diffusers import VQModel
    print("Loading VQ-VAE teacher...")
    vqvae = VQModel.from_pretrained("CompVis/ldm-celebahq-256", subfolder="vqvae")
    vqvae = vqvae.to(DEVICE).eval()
    for p in vqvae.parameters():
        p.requires_grad_(False)
    return vqvae


def cosine_loss(a, b):
    return 1.0 - F.cosine_similarity(a.flatten(1), b.flatten(1), dim=1).mean()

def decorrelation_loss(z):
    """Penalize off-diagonal channel correlations."""
    B, C, H, W = z.shape
    z_flat = z.reshape(B, C, -1)
    z_c = z_flat - z_flat.mean(dim=2, keepdim=True)
    corr = torch.bmm(z_c, z_c.transpose(1, 2)) / (H * W)
    corr = corr.mean(0)
    diag = corr.diag().sqrt().clamp(min=1e-6)
    corr = corr / (diag.unsqueeze(0) * diag.unsqueeze(1))
    mask = 1 - torch.eye(C, device=z.device)
    return (corr * mask).pow(2).sum() / (C * (C - 1))


@torch.no_grad()
def evaluate(spnn, vqvae, loader):
    spnn.eval()
    total_psnr, total_cos, n = 0, 0, 0
    for images in loader:
        images = images.to(DEVICE)
        z_s = spnn.encode(images)
        z_v = vqvae.encode(images).latents

        recon = spnn.decode(z_s)
        mse = F.mse_loss(recon.clamp(-1,1), images, reduction="none").mean(dim=(1,2,3))
        total_psnr += (-10 * torch.log10(mse)).sum().item()
        total_cos += F.cosine_similarity(z_s.flatten(1), z_v.flatten(1), dim=1).sum().item()
        n += images.shape[0]
    return {"psnr": total_psnr/n, "cos_sim": total_cos/n}


def train():
    vqvae = load_vqvae()
    train_loader, val_loader = get_loaders()

    print(f"Creating SPNNAutoencoder256v2 (cascaded)...")
    spnn = SPNNAutoencoder256(
        mix_type=MIX_TYPE, scale_bound=SCALE_BOUND,
    ).to(DEVICE)

    n_params = sum(p.numel() for p in spnn.parameters() if p.requires_grad)
    print(f"  Parameters: {n_params:,}")

    for i, b in enumerate(spnn.blocks):
        name = b.__class__.__name__
        if hasattr(b, 'in_ch'):
            print(f"  Block {i}: {name} {b.in_ch} → {b.out_ch}")
        else:
            print(f"  Block {i}: {name} (r={b.r})")

    optimizer = torch.optim.AdamW(spnn.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS * len(train_loader), eta_min=1e-6
    )

    print(f"\nTraining for {NUM_EPOCHS} epochs on {DEVICE}")
    print(f"  Phase 1 (alignment): epochs 1-{PHASE1_EPOCHS}")
    print(f"  Phase 2 (+ recon):   epochs {PHASE1_EPOCHS+1}-{NUM_EPOCHS}")
    print()

    best_psnr = 0
    best_cos = -1

    for epoch in range(NUM_EPOCHS):
        phase = 1 if epoch < PHASE1_EPOCHS else 2
        spnn.train()
        running = {"loss": 0, "l1": 0, "mse": 0, "cos": 0, "recon": 0, "decorr": 0}
        n_steps = 0

        for batch_idx, images in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
            images = images.to(DEVICE)

            with torch.no_grad():
                z_vae = vqvae.encode(images).latents

            z_spnn = spnn.encode(images)

            lat_l1 = F.l1_loss(z_spnn, z_vae)
            lat_mse = F.mse_loss(z_spnn, z_vae)
            cos = cosine_loss(z_spnn, z_vae)
            loss = W_LATENT_L1 * lat_l1 + W_LATENT_MSE * lat_mse + W_COSINE * cos

            decorr = decorrelation_loss(z_spnn)
            loss = loss + W_DECORR * decorr

            recon_loss = torch.tensor(0.0, device=DEVICE)

            if phase >= 2:
                recon = spnn.decode(z_spnn)
                recon_loss = F.l1_loss(recon, images) + F.mse_loss(recon, images)
                loss = loss + W_RECON * recon_loss

                if batch_idx % 4 == 0:
                    z_cycle = spnn.encode(recon)
                    cycle_loss = F.mse_loss(z_cycle, z_spnn.detach())
                    loss = loss + W_CYCLE * cycle_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(spnn.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            running["loss"] += loss.item()
            running["l1"] += lat_l1.item()
            running["mse"] += lat_mse.item()
            running["cos"] += cos.item()
            running["recon"] += recon_loss.item()
            running["decorr"] += decorr.item()
            n_steps += 1

            if (batch_idx + 1) % LOG_EVERY == 0:
                avg = {k: v/n_steps for k, v in running.items()}
                lr = optimizer.param_groups[0]["lr"]
                print(f"  [{batch_idx+1}/{len(train_loader)}] "
                      f"loss={avg['loss']:.4f} l1={avg['l1']:.4f} mse={avg['mse']:.5f} "
                      f"cos={avg['cos']:.4f} recon={avg['recon']:.4f} "
                      f"decorr={avg['decorr']:.5f} lr={lr:.2e}")

        val = evaluate(spnn, vqvae, val_loader)
        avg_loss = running["loss"] / max(n_steps, 1)
        print(f"Epoch {epoch+1}: loss={avg_loss:.4f}  "
              f"PSNR={val['psnr']:.2f} dB  cos_sim={val['cos_sim']:.4f}")

        # Save
        is_best_psnr = val["psnr"] > best_psnr
        is_best_cos = val["cos_sim"] > best_cos

        if is_best_psnr:
            best_psnr = val["psnr"]
        if is_best_cos:
            best_cos = val["cos_sim"]

        ckpt = {
            "epoch": epoch + 1,
            "state_dict": spnn.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "val_metrics": val,
            "best_psnr": best_psnr,
            "best_cos": best_cos,
            "config": {
                "mix_type": MIX_TYPE, "hidden": HIDDEN,
                "r_hidden": R_HIDDEN, "scale_bound": SCALE_BOUND,
            },
        }

        torch.save(ckpt, os.path.join(SAVE_DIR, "latest.pt"))
        if is_best_psnr:
            torch.save(ckpt, os.path.join(SAVE_DIR, "best_psnr.pt"))
            print(f"  ** New best PSNR: {best_psnr:.2f} dB")
        if is_best_cos:
            torch.save(ckpt, os.path.join(SAVE_DIR, "best_cos.pt"))
            print(f"  ** New best cos_sim: {best_cos:.4f}")
        if (epoch + 1) % SAVE_EVERY == 0:
            torch.save(ckpt, os.path.join(SAVE_DIR, f"epoch_{epoch+1:03d}.pt"))

        if (epoch + 1) % SAVE_EVERY == 0:
            spnn.eval()
            with torch.no_grad():
                sample = next(iter(val_loader))[:8].to(DEVICE)
                z = spnn.encode(sample)
                recon = spnn.decode(z)

                # Also show VQ-VAE recon for reference
                z_v = vqvae.encode(sample).latents
                recon_v = vqvae.decode(z_v).sample

                grid = make_grid(torch.cat([
                    (sample.clamp(-1,1)+1)/2,
                    (recon.clamp(-1,1)+1)/2,
                    (recon_v.clamp(-1,1)+1)/2,
                ]), nrow=8, padding=2)
                save_image(grid, os.path.join(SAVE_DIR, f"recon_{epoch+1:03d}.png"))
                print(f"  Saved recon grid (rows: original, SPNN v2, VQ-VAE)")

    print(f"\nDone.")
    print(f"  Best PSNR: {best_psnr:.2f} dB")
    print(f"  Best cos_sim: {best_cos:.4f}")
    print(f"  Checkpoints in {SAVE_DIR}/")


if __name__ == "__main__":
    train()

In [2]:
## Finetune LDM ###

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.utils import save_image, make_grid
from tqdm import tqdm
import os
import copy

from spnn_model import SPNNAutoencoder256

# ── Config ──────────────────────────────────────────────────

DEVICE = "cuda"
DATASET = "korexyz/celeba-hq-256x256"

SPNN_CKPT = "checkpoints_spnn/latest.pt"
SPNN_KWARGS = dict(mix_type="cayley", hidden=192, r_hidden=384, scale_bound=2.0)

BATCH_SIZE = 32
LR = 1e-6
NUM_EPOCHS = 10
SAVE_DIR = "checkpoints_ldm_finetune"
LOG_EVERY = 200
SAVE_EVERY = 10
SAMPLE_EVERY = 10

W_DENOISE = 1.0
W_DISTILL = 0.5

EMA_DECAY = 0.9999
SAMPLE_DDIM_STEPS = 200
N_SAMPLES = 8

# ────────────────────────────────────────────────────────────

os.makedirs(SAVE_DIR, exist_ok=True)
torch.manual_seed(42)

transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(256),
    transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3),
])


# ── Data ────────────────────────────────────────────────────

class HFWrap(Dataset):
    def __init__(self, ds):
        self.ds = ds
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        img = self.ds[idx]["image"]
        if img.mode != "RGB": img = img.convert("RGB")
        return transform(img)

def get_loaders():
    from datasets import load_dataset
    ds = load_dataset(DATASET, split="train")
    n = len(ds)
    n_train = int(0.95 * n)
    train_loader = DataLoader(HFWrap(ds.select(range(n_train))),
                              batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=True)
    val_loader = DataLoader(HFWrap(ds.select(range(n_train, n))),
                            batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)
    return train_loader, val_loader


# ── Load models ─────────────────────────────────────────────

def load_vqvae():
    from diffusers import VQModel
    print("Loading frozen VQ-VAE...")
    vqvae = VQModel.from_pretrained("CompVis/ldm-celebahq-256", subfolder="vqvae")
    vqvae = vqvae.to(DEVICE).eval()
    for p in vqvae.parameters():
        p.requires_grad_(False)
    return vqvae

def load_spnn():
    print(f"Loading frozen SPNN from {SPNN_CKPT}...")
    spnn = SPNNAutoencoder256(**SPNN_KWARGS).to(DEVICE).eval()
    ckpt = torch.load(SPNN_CKPT, map_location=DEVICE)
    sd = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
    spnn.load_state_dict(sd)
    for p in spnn.parameters():
        p.requires_grad_(False)
    return spnn

def load_unets():
    from diffusers import UNet2DModel
    print("Loading UNets...")
    teacher = UNet2DModel.from_pretrained("CompVis/ldm-celebahq-256", subfolder="unet")
    teacher = teacher.to(DEVICE).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)

    student = UNet2DModel.from_pretrained("CompVis/ldm-celebahq-256", subfolder="unet")
    student = student.to(DEVICE).train()

    print(f"  UNet parameters: {sum(p.numel() for p in student.parameters()):,}")
    return teacher, student

def load_schedulers():
    from diffusers import DDPMScheduler, DDIMScheduler
    train_sched = DDPMScheduler.from_pretrained("CompVis/ldm-celebahq-256", subfolder="scheduler")
    sample_sched = DDIMScheduler.from_pretrained("CompVis/ldm-celebahq-256", subfolder="scheduler")
    return train_sched, sample_sched


# ── EMA ─────────────────────────────────────────────────────

class EMA:
    def __init__(self, model, decay=0.9999):
        self.decay = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            self.shadow[k].mul_(self.decay).add_(v, alpha=1 - self.decay)

    def state_dict(self):
        return self.shadow

    def apply_to(self, model):
        model.load_state_dict(self.shadow)


# ── Sampling ────────────────────────────────────────────────

@torch.no_grad()
def sample_and_save(unet, spnn, scheduler, epoch):
    unet.eval()
    scheduler.set_timesteps(SAMPLE_DDIM_STEPS)
    z = torch.randn(N_SAMPLES, 3, 64, 64, device=DEVICE)
    for t in tqdm(scheduler.timesteps, desc="Sampling", leave=False):
        eps = unet(z, t).sample
        z = scheduler.step(eps, t, z).prev_sample
    images = spnn.decode(z)
    grid = make_grid((images.clamp(-1, 1) + 1) / 2, nrow=4, padding=2)
    path = os.path.join(SAVE_DIR, f"samples_{epoch:03d}.png")
    save_image(grid, path)
    print(f"  Saved samples: {path}")


# ── Training ────────────────────────────────────────────────

def train():
    vqvae = load_vqvae()
    spnn = load_spnn()
    teacher, student = load_unets()
    train_sched, sample_sched = load_schedulers()
    train_loader, val_loader = get_loaders()

    ema = EMA(student, decay=EMA_DECAY)
    num_timesteps = train_sched.config.num_train_timesteps

    optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=1e-4)
    lr_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS * len(train_loader), eta_min=1e-7,
    )

    print(f"\nFinetuning LDM for SPNN latents")
    print(f"  Epochs: {NUM_EPOCHS}, LR: {LR}, Batch: {BATCH_SIZE}")
    print(f"  W_DENOISE: {W_DENOISE}, W_DISTILL: {W_DISTILL}")
    print()

    best_distill = float("inf")

    for epoch in range(NUM_EPOCHS):
        student.train()
        running = {"loss": 0, "denoise": 0, "distill": 0}
        n_steps = 0

        for batch_idx, images in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")):
            images = images.to(DEVICE)
            B = images.shape[0]

            with torch.no_grad():
                z_vae = vqvae.encode(images).latents
                z_spnn = spnn.encode(images)

            t = torch.randint(0, num_timesteps, (B,), device=DEVICE)
            noise = torch.randn_like(z_spnn)

            z_spnn_noisy = train_sched.add_noise(z_spnn, noise, t)
            z_vae_noisy = train_sched.add_noise(z_vae, noise, t)

            eps_student = student(z_spnn_noisy, t).sample

            denoise_loss = F.mse_loss(eps_student, noise)

            with torch.no_grad():
                eps_teacher = teacher(z_vae_noisy, t).sample
            distill_loss = F.mse_loss(eps_student, eps_teacher)

            loss = W_DENOISE * denoise_loss + W_DISTILL * distill_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()
            lr_sched.step()
            ema.update(student)

            running["loss"] += loss.item()
            running["denoise"] += denoise_loss.item()
            running["distill"] += distill_loss.item()
            n_steps += 1

            if (batch_idx + 1) % LOG_EVERY == 0:
                avg = {k: v / n_steps for k, v in running.items()}
                lr = optimizer.param_groups[0]["lr"]
                print(f"  [{batch_idx+1}] loss={avg['loss']:.5f} "
                      f"denoise={avg['denoise']:.5f} distill={avg['distill']:.5f} "
                      f"lr={lr:.2e}")

        # Quick val
        student.eval()
        val_denoise, val_distill, val_n = 0, 0, 0
        with torch.no_grad():
            for i, images in enumerate(val_loader):
                if i >= 10: break
                images = images.to(DEVICE)
                B = images.shape[0]
                z_vae = vqvae.encode(images).latents
                z_spnn = spnn.encode(images)
                t = torch.randint(0, num_timesteps, (B,), device=DEVICE)
                noise = torch.randn_like(z_spnn)
                eps_s = student(train_sched.add_noise(z_spnn, noise, t), t).sample
                eps_t = teacher(train_sched.add_noise(z_vae, noise, t), t).sample
                val_denoise += F.mse_loss(eps_s, noise).item() * B
                val_distill += F.mse_loss(eps_s, eps_t).item() * B
                val_n += B

        vd = val_denoise / val_n
        vdi = val_distill / val_n
        avg_loss = running["loss"] / max(n_steps, 1)
        print(f"Epoch {epoch+1}: loss={avg_loss:.5f}  "
              f"val_denoise={vd:.5f}  val_distill={vdi:.5f}")

        is_best = vdi < best_distill
        if is_best:
            best_distill = vdi

        ckpt = {
            "epoch": epoch + 1,
            "unet": student.state_dict(),
            "ema": ema.state_dict(),
            "optimizer": optimizer.state_dict(),
            "best_distill": best_distill,
        }

        torch.save(ckpt, os.path.join(SAVE_DIR, "latest.pt"))
        if is_best:
            torch.save(ckpt, os.path.join(SAVE_DIR, "best.pt"))
            print(f"  ** New best distill MSE: {best_distill:.6f}")

        if (epoch + 1) % SAVE_EVERY == 0:
            torch.save(ckpt, os.path.join(SAVE_DIR, f"epoch_{epoch+1:03d}.pt"))

        if (epoch + 1) % SAMPLE_EVERY == 0:
            student_sd = copy.deepcopy(student.state_dict())
            ema.apply_to(student)
            sample_and_save(student, spnn, sample_sched, epoch + 1)
            student.load_state_dict(student_sd)

    print(f"\nDone. Best distill MSE: {best_distill:.6f}")
    print(f"Checkpoints in {SAVE_DIR}/")
    print(f"\nUsage:")
    print(f"  ckpt = torch.load('{SAVE_DIR}/best.pt')")
    print(f"  unet.load_state_dict(ckpt['ema'])  # use EMA weights")


if __name__ == "__main__":
    train()

In [3]:
## Encode decode test ##

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.utils import save_image
from diffusers import VQModel
from datasets import load_dataset
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

from spnn_model import PixelUnshuffleBlock, ConvPINNBlock


DEVICE = "cuda"
SPNN_CKPT = "checkpoints_spnn/latest.pt"
DATASET = "korexyz/celeba-hq-256x256"
NUM_CYCLES = 100
SHOW_EVERY = 10    
IMG_IDX = 70
OUT_GRID = "cycle_test.png"
OUT_PSNR = "cycle_psnr.png"
# ────────────────────────────────────────────────────────────

torch.manual_seed(42)

def to_vis(t):
    return (t.clamp(-1, 1) + 1) / 2

# ── Load models ─────────────────────────────────────────────

print("Loading VQ-VAE...")
vqvae = VQModel.from_pretrained("CompVis/ldm-celebahq-256", subfolder="vqvae")
vqvae = vqvae.to(DEVICE).eval()

print(f"Loading SPNN from {SPNN_CKPT}...")
spnn = SPNNAutoencoder256().to(DEVICE).eval()
ckpt = torch.load(SPNN_CKPT, map_location=DEVICE, weights_only=True)
spnn.load_state_dict(ckpt["state_dict"] if "state_dict" in ckpt else ckpt)

# ── Load image ──────────────────────────────────────────────

tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(256),
    transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3),
])

ds = load_dataset(DATASET, split="validation")

# ── Run cycles ──────────────────────────────────────────────

@torch.no_grad()
def run_cycles(encode_fn, decode_fn, x_orig, collect_frames=False):
    current = x_orig.clone()
    psnrs = []
    frames = [to_vis(x_orig)] if collect_frames else None

    for i in range(1, NUM_CYCLES + 1):
        z = encode_fn(current)
        current = decode_fn(z)

        mse = F.mse_loss(current.clamp(-1,1), x_orig.clamp(-1,1)).item()
        psnr = -10 * torch.log10(torch.tensor(mse)).item()
        psnrs.append(psnr)

        if collect_frames and i % SHOW_EVERY == 0:
            frames.append(to_vis(current))

    return psnrs, frames

# ── Single image grid (IMG_IDX) ─────────────────────────────

print(f"\nGrid image: index {IMG_IDX}")
img = ds[IMG_IDX]["image"].convert("RGB")
x = tf(img).unsqueeze(0).to(DEVICE)

vae_psnrs_single, vae_frames = run_cycles(
    lambda x: vqvae.encode(x).latents,
    lambda z: vqvae.decode(z).sample,
    x, collect_frames=True)
print(f"  VQ-VAE cycle 100: PSNR={vae_psnrs_single[-1]:.2f} dB")

spnn_psnrs_single, spnn_frames = run_cycles(
    lambda x: spnn.encode(x),
    lambda z: spnn.decode(z),
    x, collect_frames=True)
print(f"  SPNN   cycle 100: PSNR={spnn_psnrs_single[-1]:.2f} dB")

x_grid = x.clone()  # save for grid building later

# ── Average PSNR over ALL validation images ─────────────────

n_val = 100#len(ds)
print(f"\nRunning {NUM_CYCLES} cycles on all {n_val} validation images for PSNR graph...")

import numpy as np
all_vae = np.zeros((n_val, NUM_CYCLES))
all_spnn = np.zeros((n_val, NUM_CYCLES))

for idx in range(n_val):
    img = ds[idx]["image"].convert("RGB")
    x = tf(img).unsqueeze(0).to(DEVICE)

    vp, _ = run_cycles(
        lambda x: vqvae.encode(x).latents,
        lambda z: vqvae.decode(z).sample,
        x)
    sp, _ = run_cycles(
        lambda x: spnn.encode(x),
        lambda z: spnn.decode(z),
        x)

    all_vae[idx] = vp
    all_spnn[idx] = sp

    if (idx + 1) % 100 == 0 or idx == n_val - 1:
        print(f"  [{idx+1}/{n_val}] VQ-VAE@100={vp[-1]:.2f}  SPNN@100={sp[-1]:.2f}")

vae_psnrs = all_vae.mean(axis=0)
spnn_psnrs = all_spnn.mean(axis=0)
vae_std = all_vae.std(axis=0)
spnn_std = all_spnn.std(axis=0)

# ── Grid: 3 rows x 11 cols ─────────────────────────────────
# Row 0: Original + blanks
# Row 1: VQ-VAE at cycles 0, 10, 20, ..., 100
# Row 2: SPNN at cycles 0, 10, 20, ..., 100

ncols = len(vae_frames)  # 11 (original + 10 snapshots)
blank = torch.ones(1, 3, 256, 256)

row_orig = [to_vis(x_grid).cpu()] + [blank] * (ncols - 1)
row_vae = [f.cpu() for f in vae_frames]
row_spnn = [f.cpu() for f in spnn_frames]

grid = torch.cat([torch.cat(row_orig), torch.cat(row_vae), torch.cat(row_spnn)])
save_image(grid, OUT_GRID, nrow=ncols, padding=2, pad_value=1.0)

# Add text labels
from PIL import Image, ImageDraw, ImageFont
grid_img = Image.open(OUT_GRID)
draw = ImageDraw.Draw(grid_img)

try:
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 16)
except:
    font = ImageFont.load_default()

# Image width per cell (approximate: total width / ncols)
cell_w = grid_img.width / ncols
pad = 2  # matches save_image padding

# Column labels (cycle numbers)
col_labels = ["Original"] + [f"Cycle {i}" for i in range(SHOW_EVERY, NUM_CYCLES + 1, SHOW_EVERY)]
for col, label in enumerate(col_labels):
    x_pos = int(col * cell_w + cell_w / 2)
    # Draw on white strip above row 0
    draw.text((x_pos, 4), label, fill="black", font=font, anchor="mt")

# Row labels
row_labels = ["", "VQ-VAE", "SPNN"]
cell_h = grid_img.height / 3
for row, label in enumerate(row_labels):
    if label:
        y_pos = int(row * cell_h + 8)
        draw.text((8, y_pos), label, fill="red" if label == "VQ-VAE" else "green",
                  font=font, anchor="lt")

grid_img.save(OUT_GRID)

print(f"\nSaved grid: {OUT_GRID}")
print(f"  Row 0: Original")
print(f"  Row 1: VQ-VAE  (cycles 0, {SHOW_EVERY}, {SHOW_EVERY*2}, ..., {NUM_CYCLES})")
print(f"  Row 2: SPNN    (cycles 0, {SHOW_EVERY}, {SHOW_EVERY*2}, ..., {NUM_CYCLES})")

# ── PSNR graph ──────────────────────────────────────────────

cycles = list(range(1, NUM_CYCLES + 1))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(cycles, vae_psnrs, label="VQ-VAE", color="#e74c3c", linewidth=2)
ax.fill_between(cycles, vae_psnrs - vae_std, vae_psnrs + vae_std,
                color="#e74c3c", alpha=0.15)
ax.plot(cycles, spnn_psnrs, label="SPNN", color="#2ecc71", linewidth=2)
ax.fill_between(cycles, spnn_psnrs - spnn_std, spnn_psnrs + spnn_std,
                color="#2ecc71", alpha=0.15)

ax.set_xlabel("Encode-Decode Cycles", fontsize=13)
ax.set_ylabel("PSNR (dB)", fontsize=13)
ax.set_title(f"Cycle Consistency: Mean PSNR over {n_val} Validation Images", fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xlim(1, NUM_CYCLES)

ax.annotate(f"{vae_psnrs[-1]:.1f} dB", xy=(NUM_CYCLES, vae_psnrs[-1]),
            xytext=(-60, 10), textcoords="offset points", fontsize=11, color="#e74c3c")
ax.annotate(f"{spnn_psnrs[-1]:.1f} dB", xy=(NUM_CYCLES, spnn_psnrs[-1]),
            xytext=(-60, -15), textcoords="offset points", fontsize=11, color="#2ecc71")

plt.tight_layout()
plt.savefig(OUT_PSNR, dpi=150)
print(f"Saved PSNR graph: {OUT_PSNR}")

print(f"\nMean PSNR after {NUM_CYCLES} cycles ({n_val} images):")
print(f"  VQ-VAE: {vae_psnrs[-1]:.2f} ± {vae_std[-1]:.2f} dB")
print(f"  SPNN:   {spnn_psnrs[-1]:.2f} ± {spnn_std[-1]:.2f} dB")

In [4]:
### Diffusion from pure noise ###

In [ ]:
import torch
import math
from torchvision.utils import save_image
from diffusers import UNet2DModel, VQModel, DDIMScheduler
from tqdm import tqdm
from spnn_model import SPNNAutoencoder256
import sys
import os

# ─── Config ───
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32
IMG_SIZE = 256
DDIM_STEPS = 200
NUM_SNAPSHOTS = 10          # intermediate frames to capture
SPNN_CKPT = "checkpoints_spnn/latest.pt"
FINETUNED_CKPT = "checkpoints_ldm_finetune/latest.pt"
LINEAR_START = 0.0015
LINEAR_END = 0.0195
SEED = 42
OUT_PATH = "diffusion_compare.png"

# Parse args
i = 1
while i < len(sys.argv):
    if sys.argv[i] == "--ckpt" and i + 1 < len(sys.argv):
        FINETUNED_CKPT = sys.argv[i + 1]; i += 2
    elif sys.argv[i] == "--seed" and i + 1 < len(sys.argv):
        SEED = int(sys.argv[i + 1]); i += 2
    else:
        i += 1

torch.manual_seed(SEED)

# ─── Load SPNN ───
def load_spnn():
    spnn = SPNNAutoencoder256(mix_type="cayley", scale_bound=2.0)
    ck = torch.load(SPNN_CKPT, map_location="cpu", weights_only=False)
    sd = ck.get("state_dict", ck.get("model", ck)) if isinstance(ck, dict) else ck
    clean = {}
    for k, v in sd.items():
        nk = k
        for p in ("spnn.", "module.", "model."):
            if nk.startswith(p): nk = nk[len(p):]
        clean[nk] = v
    spnn.load_state_dict(clean, strict=False)
    return spnn.to(DEVICE).eval()

# ─── Load models ───
print("Loading VQ-VAE...")
vqvae = VQModel.from_pretrained("CompVis/ldm-celebahq-256", subfolder="vqvae")
vqvae = vqvae.to(DEVICE).eval()

print("Loading pretrained UNet...")
unet_pretrained = UNet2DModel.from_pretrained("CompVis/ldm-celebahq-256", subfolder="unet")
unet_pretrained = unet_pretrained.to(DEVICE).eval()

print("Loading finetuned UNet...")
unet_finetuned = UNet2DModel.from_pretrained("CompVis/ldm-celebahq-256", subfolder="unet")
if os.path.exists(FINETUNED_CKPT):
    ck = torch.load(FINETUNED_CKPT, map_location="cpu", weights_only=False)
    unet_finetuned.load_state_dict(ck["ema"])
    print(f"  Loaded finetuned weights from {FINETUNED_CKPT}")
else:
    print(f"  WARNING: {FINETUNED_CKPT} not found, using pretrained weights")
unet_finetuned = unet_finetuned.to(DEVICE).eval()

print("Loading SPNN...")
spnn = load_spnn()

# ─── Scheduler ───
def make_scheduler():
    sched = DDIMScheduler(
        num_train_timesteps=1000,
        beta_start=LINEAR_START,
        beta_end=LINEAR_END,
        beta_schedule="linear",
        clip_sample=False,
        set_alpha_to_one=False,
    )
    sched.set_timesteps(DDIM_STEPS)
    return sched

# ─── Compute which timestep indices to snapshot ───
def get_snapshot_indices(total_steps, num_snapshots):
    """Return indices into the timestep list to capture, evenly spaced."""
    return [int(i * (total_steps - 1) / (num_snapshots - 1)) for i in range(num_snapshots)]

# ─── Denoise with snapshots ───
@torch.no_grad()
def denoise_with_snapshots(unet, z_init, scheduler, num_snapshots):
    """
    Run DDIM denoising, capturing intermediate latent snapshots.
    Returns list of latent tensors: [z_noise, snap1, snap2, ..., z_final]
    Length = num_snapshots + 1 (noise + snapshots including final).
    """
    z = z_init.clone()
    timesteps = scheduler.timesteps
    snap_indices = get_snapshot_indices(len(timesteps), num_snapshots)

    snapshots = [z_init.clone()]  # start with pure noise
    for i, t in enumerate(timesteps):
        t_batch = t.expand(z.shape[0]).to(DEVICE)
        pred = unet(z, t_batch).sample
        z = scheduler.step(pred, t, z, eta=0.0).prev_sample

        if i in snap_indices:
            snapshots.append(z.clone())

    # Make sure final is included
    if len(timesteps) - 1 not in snap_indices:
        snapshots.append(z.clone())

    return snapshots

# ─── Decode latents to images ───
def to_vis(t):
    return (t.clamp(-1, 1) + 1) / 2

@torch.no_grad()
def decode_snapshots_vqvae(snapshots):
    """Decode each latent snapshot through VQ-VAE decoder → [N, 3, 256, 256]"""
    frames = []
    for z in snapshots:
        img = vqvae.decode(z).sample
        frames.append(to_vis(img))
    return frames

@torch.no_grad()
def decode_snapshots_spnn(snapshots):
    """Decode each latent snapshot through SPNN decoder → [N, 3, 256, 256]"""
    frames = []
    for z in snapshots:
        img = spnn.decode(z)
        frames.append(to_vis(img))
    return frames

# ─── Generate ───
print(f"\nGenerating with DDIM ({DDIM_STEPS} steps, {NUM_SNAPSHOTS} snapshots)...")
print(f"Seed: {SEED}")

# Same starting noise for all three
import time
generator = torch.manual_seed(int(time.time()))
z_noise = torch.randn((1, 3, 64, 64), generator=generator, dtype=DTYPE).to(DEVICE)

# Row 0: VQ-VAE + pretrained LDM
print("  Row 0: VQ-VAE + pretrained LDM...")
sched0 = make_scheduler()
snaps0 = denoise_with_snapshots(unet_pretrained, z_noise, sched0, NUM_SNAPSHOTS)
row0 = decode_snapshots_vqvae(snaps0)

# Row 1: SPNN + pretrained LDM (no finetune)
print("  Row 1: SPNN + pretrained LDM (no finetune)...")
sched1 = make_scheduler()
snaps1 = denoise_with_snapshots(unet_pretrained, z_noise, sched1, NUM_SNAPSHOTS)
row1 = decode_snapshots_spnn(snaps1)

# Row 2: SPNN + finetuned LDM
print("  Row 2: SPNN + finetuned LDM...")
sched2 = make_scheduler()
snaps2 = denoise_with_snapshots(unet_finetuned, z_noise, sched2, NUM_SNAPSHOTS)
row2 = decode_snapshots_spnn(snaps2)

# ─── Build grid ───
# Each row has NUM_SNAPSHOTS + 1 images (noise + snapshots)
ncols = len(row0)
print(f"\nGrid: 3 rows × {ncols} columns")

row0_t = torch.cat(row0, dim=0)  # [ncols, 3, 256, 256]
row1_t = torch.cat(row1, dim=0)
row2_t = torch.cat(row2, dim=0)
grid = torch.cat([row0_t, row1_t, row2_t], dim=0)  # [3*ncols, 3, 256, 256]

save_image(grid, OUT_PATH, nrow=ncols, padding=3, pad_value=1.0)

print(f"\nSaved → {OUT_PATH}")
print(f"  Row 0: VQ-VAE decoder  + pretrained UNet  (baseline)")
print(f"  Row 1: SPNN decoder    + pretrained UNet   (shows latent mismatch)")
print(f"  Row 2: SPNN decoder    + finetuned UNet    (after adaptation)")
print(f"  Columns: pure noise → intermediate denoising steps → final")

In [5]:
## Inpainting DDNM

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.utils import save_image, make_grid
import os
import numpy as np

from spnn_model import SPNNAutoencoder256

DEVICE = "cuda"
DATASET = "korexyz/celeba-hq-256x256"
VAL_IMAGE_IDX = 19

SPNN_CKPT = "checkpoints_spnn/latest.pt"
LDM_FINETUNE_CKPT = "checkpoints_ldm_finetune/latest.pt"

DDIM_STEPS = 200          # number of reverse diffusion steps
ETA = 0.0                 # 0 = deterministic DDIM, 1 = full stochastic DDPM
OUT_DIR = "ddnm_out"

SPNN_KWARGS = dict(mix_type="cayley", scale_bound=2.0)

os.makedirs(OUT_DIR, exist_ok=True)

TRANSFORM = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3),
])


def to_img(t):
    """[-1,1] tensor -> [0,1] for saving."""
    return (t.clamp(-1, 1) + 1) / 2



def load_image():
    from datasets import load_dataset
    ds = load_dataset(DATASET, split="validation")
    img = ds[VAL_IMAGE_IDX]["image"]
    if img.mode != "RGB":
        img = img.convert("RGB")
    return TRANSFORM(img).unsqueeze(0).to(DEVICE)


def load_vqvae():
    from diffusers import VQModel
    print("Loading VQ-VAE ...")
    vqvae = VQModel.from_pretrained("CompVis/ldm-celebahq-256", subfolder="vqvae")
    vqvae.to(DEVICE).eval()
    for p in vqvae.parameters():
        p.requires_grad_(False)
    return vqvae


def load_spnn():
    print(f"Loading SPNN from {SPNN_CKPT} ...")
    model = SPNNAutoencoder256(**SPNN_KWARGS).to(DEVICE).eval()
    ckpt = torch.load(SPNN_CKPT, map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt.get("state_dict", ckpt))
    for p in model.parameters():
        p.requires_grad_(False)
    return model


def load_unet(ckpt_path=None):
    from diffusers import UNet2DModel
    tag = "pretrained" if ckpt_path is None else f"finetuned ({ckpt_path})"
    print(f"Loading UNet [{tag}] ...")
    unet = UNet2DModel.from_pretrained("CompVis/ldm-celebahq-256", subfolder="unet")
    if ckpt_path is not None:
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
        state = ck.get("ema", ck.get("unet", ck))
        unet.load_state_dict(state)
    unet.to(DEVICE).eval()
    for p in unet.parameters():
        p.requires_grad_(False)
    return unet


def load_scheduler():
    from diffusers import DDIMScheduler
    return DDIMScheduler.from_pretrained("CompVis/ldm-celebahq-256", subfolder="scheduler")



def make_masks(h=256, w=256, f=4):
    """
    Returns {name: (pixel_mask, latent_mask)}.
    mask = 1 where we want to inpaint (unknown / missing region).
    """
    lh, lw = h // f, w // f
    masks = {}

    # Center box ~25 %
    m = torch.zeros(1, 1, h, w)
    s = h // 4
    m[:, :, s:s + h // 2, s:s + w // 2] = 1
    ml = torch.zeros(1, 1, lh, lw)
    ml[:, :, s // f:(s + h // 2) // f, s // f:(s + w // 2) // f] = 1
    masks["center_box"] = (m, ml)

    # Eyes
    m = torch.zeros(1, 1, h, w)
    m[:, :, 70:130, 40:216] = 1
    ml = F.interpolate(m, size=(lh, lw), mode="nearest")
    masks["eyes"] = (m, ml)

    # Lower face
    m = torch.zeros(1, 1, h, w)
    m[:, :, 140:240, 30:226] = 1
    ml = F.interpolate(m, size=(lh, lw), mode="nearest")
    masks["lower_face"] = (m, ml)

    # Random irregular (brush strokes)
    rng = np.random.RandomState(123)
    m = torch.zeros(1, 1, h, w)
    for _ in range(8):
        x, y = rng.randint(30, w - 30), rng.randint(30, h - 30)
        for _ in range(60):
            r = rng.randint(8, 20)
            m[:, :, max(0, y - r):min(h, y + r), max(0, x - r):min(w, x + r)] = 1
            x = int(np.clip(x + rng.randint(-12, 13), 0, w - 1))
            y = int(np.clip(y + rng.randint(-12, 13), 0, h - 1))
    ml = F.interpolate(m, size=(lh, lw), mode="nearest")
    masks["irregular"] = (m, ml)

    return masks



@torch.no_grad()
def ddnm_inpaint(encode_fn, decode_fn, unet, scheduler, x_orig, mask_latent):
    """
    DDNM inpainting in latent space.

    For the inpainting degradation A = diag(1 - mask), the DDNM update is:
        x0_hat = A^+ y + (I - A^+ A) x0_pred
               = (1 - mask) * z_orig  +  mask * x0_pred

    At each DDIM reverse step we:
      1. Predict x0 from current z_t via the UNet noise prediction.
      2. Apply the DDNM null-space projection to get x0_hat.
      3. Compute z_{t-1} using the DDIM update with x0_hat.

    Args:
        encode_fn:   image [1,3,256,256] -> latent [1,3,64,64]
        decode_fn:   latent [1,3,64,64] -> image [1,3,256,256]
        unet:        diffusion UNet
        scheduler:   DDIMScheduler (already configured)
        x_orig:      original image [1,3,256,256]
        mask_latent: binary mask [1,1,64,64], 1 = inpaint region

    Returns:
        decoded result [1,3,256,256]
    """
    scheduler.set_timesteps(DDIM_STEPS)
    timesteps = scheduler.timesteps

    # Encode original image to get the known latent values
    z_orig = encode_fn(x_orig)                           # [1, 3, 64, 64]

    # Known-region mask (complement of inpaint mask)
    known = 1.0 - mask_latent                            # 1 where pixels are known

    # Start from pure noise
    z_t = torch.randn_like(z_orig)

    alphas_cumprod = scheduler.alphas_cumprod.to(DEVICE)

    for i, t in enumerate(timesteps):
        # ── 1. Predict x0 from z_t ──
        noise_pred = unet(z_t, t).sample

        # x0_pred = (z_t - sqrt(1 - alpha_bar_t) * eps) / sqrt(alpha_bar_t)
        alpha_bar_t = alphas_cumprod[t]
        sqrt_alpha_bar = alpha_bar_t.sqrt()
        sqrt_one_minus_alpha_bar = (1 - alpha_bar_t).sqrt()
        x0_pred = (z_t - sqrt_one_minus_alpha_bar * noise_pred) / sqrt_alpha_bar

        # ── 2. DDNM projection ──
        # Replace known region with ground-truth latent, keep predicted in unknown
        x0_hat = known * z_orig + mask_latent * x0_pred

        # ── 3. Compute z_{t-1} via DDIM ──
        if i < len(timesteps) - 1:
            t_prev = timesteps[i + 1]
            alpha_bar_prev = alphas_cumprod[t_prev]
        else:
            alpha_bar_prev = torch.tensor(1.0, device=DEVICE)

        sqrt_alpha_bar_prev = alpha_bar_prev.sqrt()
        sqrt_one_minus_alpha_bar_prev = (1 - alpha_bar_prev).sqrt()

        # Recompute epsilon from x0_hat (not x0_pred) for consistency
        eps_hat = (z_t - sqrt_alpha_bar * x0_hat) / sqrt_one_minus_alpha_bar

        # DDIM deterministic step (eta = 0)
        if ETA > 0 and i < len(timesteps) - 1:
            sigma = (ETA
                     * ((1 - alpha_bar_prev) / (1 - alpha_bar_t)).sqrt()
                     * (1 - alpha_bar_t / alpha_bar_prev).sqrt())
            z_t = (sqrt_alpha_bar_prev * x0_hat
                   + (1 - alpha_bar_prev - sigma ** 2).sqrt() * eps_hat
                   + sigma * torch.randn_like(z_t))
        else:
            z_t = sqrt_alpha_bar_prev * x0_hat + sqrt_one_minus_alpha_bar_prev * eps_hat

    # Final projection (clean)
    z_final = known * z_orig + mask_latent * z_t

    return decode_fn(z_final)



def psnr(x, ref, mask=None):
    """Compute PSNR in [-1,1] range. If mask given, only over mask=1 pixels."""
    x = x.clamp(-1, 1)
    ref = ref.clamp(-1, 1)
    if mask is not None:
        n = mask.sum() * x.shape[1]  # channels
        mse = ((x - ref) ** 2 * mask).sum() / n
    else:
        mse = F.mse_loss(x, ref)
    if mse < 1e-10:
        return 99.0
    return -10 * torch.log10(mse).item()



def main():
    # Load everything
    image = load_image()
    vqvae = load_vqvae()
    spnn = load_spnn()
    unet_pre = load_unet()

    try:
        unet_ft = load_unet(LDM_FINETUNE_CKPT)
        has_ft = True
    except Exception as e:
        print(f"[WARN] Finetuned UNet not found ({e}), skipping pipeline C")
        has_ft = False

    scheduler = load_scheduler()
    masks = make_masks()

    # Encoder / decoder callables
    def vae_enc(x):
        return vqvae.encode(x).latents

    def vae_dec(z):
        return vqvae.decode(z).sample

    def spnn_enc(x):
        return spnn.encode(x)

    def spnn_dec(z):
        return spnn.decode(z)

    print(f"\nDDNM inpainting  |  image idx {VAL_IMAGE_IDX}  |  DDIM steps {DDIM_STEPS}  |  eta {ETA}")
    print(f"Masks: {list(masks.keys())}\n")

    for mask_name, (mask_px, mask_lt) in masks.items():
        pct = mask_px.sum().item() / (256 * 256) * 100
        print(f"{'='*55}")
        print(f"Mask: {mask_name}  ({pct:.1f}% masked)")
        print(f"{'='*55}")

        mask_px = mask_px.to(DEVICE)
        mask_lt = mask_lt.to(DEVICE)
        unmask_px = 1 - mask_px

        # Masked input visualization (gray fill)
        masked_vis = image * unmask_px + mask_px * 0.5

        # Run pipelines
        pipelines = {}

        print("  [A] VQ-VAE + pretrained LDM ...")
        pipelines["VAE+LDM"] = ddnm_inpaint(vae_enc, vae_dec, unet_pre, scheduler, image, mask_lt)

        print("  [B] SPNN + pretrained LDM ...")
        pipelines["SPNN+LDM"] = ddnm_inpaint(spnn_enc, spnn_dec, unet_pre, scheduler, image, mask_lt)

        if has_ft:
            print("  [C] SPNN + finetuned LDM ...")
            pipelines["SPNN+ftLDM"] = ddnm_inpaint(spnn_enc, spnn_dec, unet_ft, scheduler, image, mask_lt)

        # ── Metrics ──
        print()
        for tag, res in pipelines.items():
            # Pixel-blend for clean comparison
            blended = mask_px * res + unmask_px * image

            p_full = psnr(blended, image)
            p_mask = psnr(res, image, mask_px)
            p_unmask = psnr(res, image, unmask_px)

            print(f"    {tag:>14s}  |  full PSNR {p_full:6.2f}  |  "
                  f"masked-region {p_mask:6.2f}  |  unmasked (seam) {p_unmask:6.2f}")

        # ── Save grids ──
        # Blended grid (paste original outside mask)
        row_blend = [to_img(image), to_img(masked_vis)]
        labels = ["Original", "Masked"]
        for tag, res in pipelines.items():
            row_blend.append(to_img(mask_px * res + unmask_px * image))
            labels.append(tag)

        grid = make_grid(torch.cat(row_blend), nrow=len(row_blend), padding=4, pad_value=1)
        p = os.path.join(OUT_DIR, f"ddnm_{mask_name}.png")
        save_image(grid, p)
        print(f"    -> {p}  [{', '.join(labels)}]")

        # Raw grid (full decoder output, no pixel blending)
        row_raw = [to_img(image), to_img(masked_vis)]
        for res in pipelines.values():
            row_raw.append(to_img(res))
        grid_raw = make_grid(torch.cat(row_raw), nrow=len(row_raw), padding=4, pad_value=1)
        p_raw = os.path.join(OUT_DIR, f"ddnm_{mask_name}_raw.png")
        save_image(grid_raw, p_raw)
        print(f"    -> {p_raw}  [raw decoder output]")
        print()

    # ── Cycle consistency on inpainted result ────────────────
    print("=" * 55)
    print("Cycle consistency test on inpainted output (center_box)")
    print("  Encode -> decode 10x, measure drift from cycle-0.")
    print("=" * 55)

    mask_px, mask_lt = masks["center_box"]
    mask_px, mask_lt = mask_px.to(DEVICE), mask_lt.to(DEVICE)

    res_vae = ddnm_inpaint(vae_enc, vae_dec, unet_pre, scheduler, image, mask_lt)
    if has_ft:
        res_spnn = ddnm_inpaint(spnn_enc, spnn_dec, unet_ft, scheduler, image, mask_lt)
    else:
        res_spnn = ddnm_inpaint(spnn_enc, spnn_dec, unet_pre, scheduler, image, mask_lt)

    x_v = res_vae.clone()
    x_s = res_spnn.clone()

    print(f"\n  {'Cycle':>6}  {'VAE PSNR':>10}  {'SPNN PSNR':>11}")
    print("  " + "-" * 32)

    for c in range(1, 11):
        x_v = vae_dec(vae_enc(x_v))
        x_s = spnn_dec(spnn_enc(x_s))
        pv = psnr(x_v, res_vae)
        ps = psnr(x_s, res_spnn)
        print(f"  {c:>6d}  {pv:>10.2f}  {ps:>11.2f}")

    # Save cycle comparison (2x2: start | after-10-cycles)
    grid_cycle = make_grid(torch.cat([
        to_img(res_vae), to_img(x_v),
        to_img(res_spnn), to_img(x_s),
    ]), nrow=2, padding=4, pad_value=1)
    p_cyc = os.path.join(OUT_DIR, "ddnm_cycle_comparison.png")
    save_image(grid_cycle, p_cyc)
    print(f"\n  -> {p_cyc}  [top: VAE start/10cyc, bottom: SPNN start/10cyc]")

    print(f"\nAll outputs saved to {OUT_DIR}/")


if __name__ == "__main__":
    main()